# EDA: Motor Insurance Claim Severity

This notebook is the first step of a GLM vs. XGBoost comparative analysis on motor insurance claim data collected from Ethio-Insurance. The goal is to characterise the response distribution, assess each candidate predictor, document keep/drop decisions, and pass a well-justified predictor set to the modelling notebook.

**Dataset:** `final_merged_insurance_data_updated.csv`  
**Response:** `total_claim_severity` (Birr)  
**Outputs:** 11 diagnostic plots saved to `eda_output/`

In [1]:
# 01_eda.py — Consolidated Exploratory Data Analysis (all 4 parts)
# INPUT:  final_merged_insurance_data_updated.csv (same directory)
# OUTPUT: eda_output/ folder with 11 PNG plots + printed statistics
"""
EDA: Dataset Overview, Dependent Variable, Categorical & Continuous Predictors,
Correlation, Branch Analysis, Predictor Decisions, ML Recommendation
Thesis: Predicting Motor Insurance Claim Severity | Addis Ababa University
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
import os, sys, io, warnings
warnings.filterwarnings('ignore')
#sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
if hasattr(sys.stdout, 'buffer'):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')


DATA_PATH = 'final_merged_insurance_data_updated.csv'
OUTPUT_DIR = 'eda_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams.update({
    'figure.figsize': (12, 6), 'font.size': 11,
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'figure.dpi': 150, 'savefig.bbox': 'tight'
})
sns.set_style('whitegrid')

df = pd.read_csv(DATA_PATH)
TARGET = 'total_claim_severity'
CATEGORICAL = ['class_of_business', 'vehicle_type_group', 'accident_type', 'branch_name', 'policy_type']
CONTINUOUS = ['sum_insured_numeric', 'reporting_lag_days', 'coverage_duration_days']

## 1. Dataset Overview and Response Variable

I start by confirming data shape, column types, and missing values before examining the response. Motor insurance claim severity is characteristically right-skewed with a heavy upper tail — understanding that shape early on informs every distributional and link-function choice that follows.

The key question here is whether `total_claim_severity` is better modelled under a Gamma or Tweedie GLM, and whether an XGBoost `reg:gamma` objective is appropriate. The severity component breakdown (OD vs. TP-BI vs. TP-PD) also matters for interpreting coefficient signs in the final model.

In [2]:
# ---
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn Names and Types:")
print("-" * 50)
for col in df.columns:
    print(f"  {col:<30} {str(df[col].dtype):<15} {df[col].notna().sum()}/{len(df)} non-null")

print("MISSING VALUES SUMMARY")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
miss_df = miss_df[miss_df['Missing'] > 0].sort_values('Missing', ascending=False)
if len(miss_df) > 0:
    print(miss_df.to_string())
else:
    print("No missing values in any column.")

print("VARIABLE ROLES")
print(f"  Target Variable:       {TARGET}")
print(f"  Categorical Predictors: {CATEGORICAL}")
print(f"  Continuous Predictors:  {CONTINUOUS}")
print(f"  Binary Predictor:       is_comprehensive")

severity = df[TARGET]
print(f"\n  Count:           {severity.count():,.0f}")
print(f"  Mean:            {severity.mean():,.2f} Birr")
print(f"  Median:          {severity.median():,.2f} Birr")
print(f"  Std Dev:         {severity.std():,.2f} Birr")
print(f"  Min:             {severity.min():,.2f} Birr")
print(f"  Max:             {severity.max():,.2f} Birr")
print(f"  Skewness:        {severity.skew():.4f}")
print(f"  Kurtosis:        {severity.kurtosis():.4f}")

print(f"\n  Percentiles:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"    {p}th:  {severity.quantile(p/100):>15,.2f} Birr")
print(f"\n  Mean/Median Ratio: {severity.mean()/severity.median():.2f}x")
print(f"  -> Confirms strong right-skew (typical of insurance severity data)")

# Side-by-side histograms on raw and log scale: the gap between mean and median
# on the raw scale quantifies right-skew; the log panel assesses approximate
# log-normality, which would support a log-OLS baseline but is not required for
# Gamma GLM — the family only needs E[Y|X] to follow a log-linear structure.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(severity, bins=80, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(severity.mean(), color='red', ls='--', lw=1.5, label=f'Mean: {severity.mean():,.0f}')
axes[0].axvline(severity.median(), color='orange', ls='--', lw=1.5, label=f'Median: {severity.median():,.0f}')
axes[0].set_title('Distribution of Total Claim Severity')
axes[0].set_xlabel('Total Claim Severity (Birr)'); axes[0].set_ylabel('Frequency'); axes[0].legend(fontsize=9)

log_sev = df['log_claim_severity']
axes[1].hist(log_sev, bins=50, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[1].axvline(log_sev.mean(), color='red', ls='--', lw=1.5, label=f'Mean: {log_sev.mean():.2f}')
axes[1].axvline(log_sev.median(), color='orange', ls='--', lw=1.5, label=f'Median: {log_sev.median():.2f}')
axes[1].set_title('Distribution of Log(Claim Severity)')
axes[1].set_xlabel('Log(Total Claim Severity)'); axes[1].set_ylabel('Frequency'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '01_severity_distribution.png')); plt.close()

# Box plots on both scales: raw scale exposes the upper-tail outlier extent;
# log scale preserves the interquartile spread, making it meaningful for
# cross-group comparisons in subsequent cells.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(severity, vert=True, patch_artist=True, boxprops=dict(facecolor='#2196F3', alpha=0.6))
axes[0].set_title('Box Plot: Total Claim Severity'); axes[0].set_ylabel('Claim Severity (Birr)')
axes[0].ticklabel_format(style='plain', axis='y')
axes[1].boxplot(log_sev, vert=True, patch_artist=True, boxprops=dict(facecolor='#4CAF50', alpha=0.6))
axes[1].set_title('Box Plot: Log(Claim Severity)'); axes[1].set_ylabel('Log(Claim Severity)')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '02_severity_boxplot.png')); plt.close()

# Q-Q plot of log(severity) against the Normal reference line.
# Systematic departure in the tails — typical for insurance losses — supports
# Gamma GLM over OLS on the log-transformed response, which would require
# approximately normal residuals on the log scale.
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
stats.probplot(log_sev.dropna(), dist="norm", plot=ax)
ax.set_title('Q-Q Plot: Log(Claim Severity) vs Normal Distribution')
ax.get_lines()[0].set(color='#2196F3', markersize=3); ax.get_lines()[1].set(color='red', linewidth=1.5)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '03_severity_qqplot.png')); plt.close()

# Decomposing total severity into OD, TP-BI, and TP-PD: if one component
# dominates both claim count and total amount, a single-response GLM is
# justified without sub-model disaggregation by coverage type.
od_sum = df['od_total'].sum(); tp_bi_sum = df['tp_bi_total'].sum(); tp_pd_sum = df['tp_pd_total'].sum()
total = od_sum + tp_bi_sum + tp_pd_sum
print(f"\n  Severity Component Breakdown:")
print(f"    Own Damage (OD):         {od_sum:>15,.2f} Birr  ({od_sum/total*100:.1f}%)")
print(f"    TP Bodily Injury (BI):   {tp_bi_sum:>15,.2f} Birr  ({tp_bi_sum/total*100:.1f}%)")
print(f"    TP Property Damage (PD): {tp_pd_sum:>15,.2f} Birr  ({tp_pd_sum/total*100:.1f}%)")

od_claims = (df['od_total'] > 0).sum(); bi_claims = (df['tp_bi_total'] > 0).sum(); pd_claims = (df['tp_pd_total'] > 0).sum()
print(f"\n  Claims with each component:")
print(f"    OD > 0:    {od_claims} ({od_claims/len(df)*100:.1f}%)")
print(f"    TP_BI > 0: {bi_claims} ({bi_claims/len(df)*100:.1f}%)")
print(f"    TP_PD > 0: {pd_claims} ({pd_claims/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = ['Own Damage', 'TP Bodily Injury', 'TP Property Damage']
sizes = [od_sum, tp_bi_sum, tp_pd_sum]; colors_pie = ['#2196F3', '#FF5722', '#FFC107']
axes[0].pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Severity by Component (Total Amount)')
comp_counts = [od_claims, bi_claims, pd_claims]
bars = axes[1].bar(labels, comp_counts, color=colors_pie, edgecolor='white')
axes[1].set_title('Number of Claims with Each Component'); axes[1].set_ylabel('Number of Claims')
for bar, count in zip(bars, comp_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f'{count}', ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '04_severity_components.png')); plt.close()

# Shapiro-Wilk on a subsample of log(severity): rejection even after the
# log-transform rules out log-OLS and reinforces the Gamma GLM — the Gamma
# family requires only E[Y|X] = exp(Xβ), not normality of log(Y).
stat, p_value = stats.shapiro(log_sev.dropna().sample(min(5000, len(log_sev)), random_state=42))
print(f"\n  Normality Test on Log(Severity): Shapiro-Wilk stat={stat:.4f}, p={p_value:.6f}")
if p_value < 0.05:
    print(f"    -> Log-severity is NOT normally distributed -> supports Gamma GLM")

# Gamma distribution assumes Var(Y) ∝ μ² (constant coefficient of variation).
# A strong positive mean-variance relationship across deciles confirms this;
# if the relationship were Var ∝ μ instead, Poisson/Tweedie would be more appropriate.
severity_clean = severity.dropna()
decile_labels = pd.qcut(severity_clean, 10, labels=False, duplicates='drop')
means = severity_clean.groupby(decile_labels).mean()
variances = severity_clean.groupby(decile_labels).var()
corr_mv = np.corrcoef(means, variances)[0, 1]
print(f"\n  Mean-Variance Correlation: {corr_mv:.4f}")
print(f"  -> {'Supports Gamma distribution assumption' if corr_mv > 0.7 else 'Consider alternative distributions'}")


Shape: 2586 rows x 23 columns

Column Names and Types:
--------------------------------------------------
  claim_no                       object          2586/2586 non-null
  policy_no                      object          2427/2586 non-null
  plate_no                       object          2586/2586 non-null
  source_file                    object          2586/2586 non-null
  class_of_business              object          2586/2586 non-null
  vehicle_type_group             object          2586/2586 non-null
  vehicle_type                   object          2586/2586 non-null
  accident_type                  object          2586/2586 non-null
  branch_name                    object          2586/2586 non-null
  sum_insured_numeric            float64         2582/2586 non-null
  reporting_lag_days             float64         2371/2586 non-null
  coverage_duration_days         float64         2436/2586 non-null
  is_comprehensive               int64           2586/2586 non-null
  policy_

## 2. Categorical Predictors

We examine four categorical variables — class of business, vehicle type group, accident type, and policy type — through frequency tables, within-group severity distributions, and Kruskal-Wallis significance tests. Branch name is analysed separately because its 43 unique levels raise a model-complexity question that the others do not.

The Kruskal-Wallis test is used throughout in place of one-way ANOVA: claim severity within any category is far from normally distributed, making the F-statistic's normality assumption untenable.

In [3]:
# ---
categoricals = ['class_of_business', 'vehicle_type_group', 'accident_type', 'policy_type']
for col in categoricals:
    print(f"\n--- {col} ---")
    vc = df[col].value_counts(); pct = df[col].value_counts(normalize=True) * 100
    summary = pd.DataFrame({'Count': vc, 'Percent': pct.round(1)})
    print(summary.to_string())

# Frequency counts to flag thin categories before modeling — levels with fewer
# than ~30 observations will produce high-variance GLM coefficients and should
# be merged with a reference category before dummy encoding.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors8 = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4', '#795548', '#607D8B']
for idx, col in enumerate(categoricals):
    ax = axes[idx // 2][idx % 2]; vc = df[col].value_counts()
    bars = ax.bar(range(len(vc)), vc.values, color=colors8[:len(vc)], edgecolor='white')
    ax.set_xticks(range(len(vc))); ax.set_xticklabels(vc.index, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{col}', fontsize=13, fontweight='bold'); ax.set_ylabel('Count')
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(val), ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '05_categorical_frequencies.png')); plt.close()

# Within-group descriptive statistics: a large mean/median ratio within a
# category signals within-group right-skew consistent with the Gamma assumption.
# Groups with high means but similar medians likely contain a small number of
# extremely large claims that a Gamma GLM handles better than OLS.
for col in categoricals:
    print(f"\n--- Severity by {col} ---")
    grp = df.groupby(col)[TARGET].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    grp['mean'] = grp['mean'].map('{:,.0f}'.format); grp['median'] = grp['median'].map('{:,.0f}'.format)
    grp['std'] = grp['std'].map('{:,.0f}'.format)
    grp['min'] = grp['min'].map('{:,.0f}'.format); grp['max'] = grp['max'].map('{:,.0f}'.format)
    print(grp.to_string())

# Log-scale boxplots ordered by group median: on raw scale, a single extreme
# claim per category collapses the visual and hides between-group differences.
# Log-scale preserves the distributional signal that the Kruskal-Wallis test
# below formalises.
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for idx, col in enumerate(categoricals):
    ax = axes[idx // 2][idx % 2]
    order = df.groupby(col)[TARGET].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='log_claim_severity', order=order, palette='Set2', ax=ax, fliersize=2)
    ax.set_title(f'Log(Severity) by {col}', fontsize=13, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('Log(Claim Severity)'); ax.tick_params(axis='x', rotation=45, labelsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '06_severity_by_categorical.png')); plt.close()

# Horizontal bars ranked by median severity: median is preferred over mean here
# because the heavy upper tail makes the mean sensitive to a handful of extreme
# claims per group, distorting the empirical ranking of group effects.
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for idx, col in enumerate(categoricals):
    ax = axes[idx // 2][idx % 2]
    medians = df.groupby(col)[TARGET].median().sort_values(ascending=False)
    bars = ax.barh(range(len(medians)), medians.values, color=colors8[:len(medians)], edgecolor='white')
    ax.set_yticks(range(len(medians))); ax.set_yticklabels(medians.index, fontsize=9)
    ax.set_title(f'Median Severity by {col}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Median Claim Severity (Birr)'); ax.ticklabel_format(style='plain', axis='x')
    for bar, val in zip(bars, medians.values):
        ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2, f'{val:,.0f}', ha='left', va='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '07_median_severity_by_categorical.png')); plt.close()

# Kruskal-Wallis test for equal severity distributions across categories.
# ANOVA is invalid here because within-group normality is clearly violated;
# the H-statistic tests whether rank distributions across groups are exchangeable.
# Significance at p<0.001 justifies including the variable as a predictor.
for col in categoricals:
    groups = [g[TARGET].values for _, g in df.groupby(col)]
    if len(groups) >= 2:
        h_stat, p_val = stats.kruskal(*groups)
        sig = "*** SIGNIFICANT" if p_val < 0.001 else "** SIGNIFICANT" if p_val < 0.01 else "* SIGNIFICANT" if p_val < 0.05 else "NOT significant"
        print(f"  {col:<25} H={h_stat:>10.2f}   p={p_val:.6f}   {sig}")

# Branch is analysed separately: 43 unique levels would generate 42 dummy
# predictors in the GLM, inflating model degrees of freedom substantially.
# The eta² effect size quantifies the explanatory gain; a small effect (η² < 0.06)
# does not justify that cost.
branch_counts = df['branch_name'].value_counts()
print(f"\n  Total unique branches: {len(branch_counts)}")
small_threshold = 30
small_branches = branch_counts[branch_counts < small_threshold]
print(f"  Branches with < {small_threshold} observations: {len(small_branches)}")
branch_groups = [g[TARGET].values for _, g in df.groupby('branch_name')]
h_stat, p_val = stats.kruskal(*branch_groups)
n = len(df); k = len(branch_counts)
eta_sq = (h_stat - k + 1) / (n - k)
print(f"  Kruskal-Wallis: H={h_stat:.2f}, p={p_val:.6f}")
print(f"  Effect size (eta-squared): {eta_sq:.4f} ({'Small' if eta_sq < 0.06 else 'Medium' if eta_sq < 0.14 else 'Large'})")
print(f"  DECISION: DROP — 43 categories, small effect, too many dummies for GLM")

# Volume and median severity by branch: the red threshold line marks the
# minimum count for a stable GLM coefficient estimate. Branches below it
# cannot be reliably estimated as separate levels.
top_n = 20; top_branches = branch_counts.head(top_n)
other_count = branch_counts.iloc[top_n:].sum() if len(branch_counts) > top_n else 0
fig, axes = plt.subplots(1, 2, figsize=(16, 10))
if other_count > 0:
    plot_labels = list(top_branches.index) + [f'Other ({len(branch_counts)-top_n} branches)']
    plot_values = list(top_branches.values) + [other_count]
else:
    plot_labels = list(branch_counts.index); plot_values = list(branch_counts.values)
bar_colors = cm.tab20(np.linspace(0, 1, len(plot_labels)))
axes[0].barh(range(len(plot_labels)), plot_values, color=bar_colors, edgecolor='white')
axes[0].set_yticks(range(len(plot_labels))); axes[0].set_yticklabels(plot_labels, fontsize=8)
axes[0].set_title('Number of Claims by Branch (Top 20)', fontweight='bold'); axes[0].set_xlabel('Count')
axes[0].axvline(x=small_threshold, color='red', ls='--', lw=1, label=f'Threshold ({small_threshold})')
axes[0].legend(fontsize=8); axes[0].invert_yaxis()
for i, v in enumerate(plot_values): axes[0].text(v + 2, i, str(v), va='center', fontsize=7)
top_branch_names = top_branches.index
branch_medians = df[df['branch_name'].isin(top_branch_names)].groupby('branch_name')[TARGET].median()
branch_medians = branch_medians.reindex(top_branch_names).sort_values(ascending=True)
axes[1].barh(range(len(branch_medians)), branch_medians.values, color='#4CAF50', edgecolor='white')
axes[1].set_yticks(range(len(branch_medians))); axes[1].set_yticklabels(branch_medians.index, fontsize=8)
axes[1].set_title('Median Claim Severity by Branch (Top 20)', fontweight='bold')
axes[1].set_xlabel('Median Severity (Birr)'); axes[1].ticklabel_format(style='plain', axis='x')
for i, v in enumerate(branch_medians.values): axes[1].text(v + 1000, i, f'{v:,.0f}', va='center', fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '08_branch_analysis.png')); plt.close()


--- class_of_business ---
                   Count  Percent
class_of_business                
Commercial          1465     56.7
Private              629     24.3
Public_Transport     492     19.0

--- vehicle_type_group ---
                    Count  Percent
vehicle_type_group                
Automobile           1027     39.7
Truck                 838     32.4
Bus_Minibus           285     11.0
Not_Specified         250      9.7
Pick_Up               155      6.0
Motorcycle             23      0.9
Special_Purpose         8      0.3

--- accident_type ---
                Count  Percent
accident_type                 
Collision        2292     88.6
Overturning       209      8.1
Theft              24      0.9
Breaking_Glass     21      0.8
Other              20      0.8
Fire               12      0.5
Flying_Object       8      0.3

--- policy_type ---
               Count  Percent
policy_type                  
COMPREHENSIVE   1623     62.8
TP_ONLY          963     37.2

--- Severity by 

## 3. Continuous Predictors and Correlation Structure

Three continuous predictors are assessed: sum insured, reporting lag, and coverage duration. The analysis checks distributional shape, strength of association with severity via Spearman rank correlation, and pairwise collinearity among predictors.

Spearman's ρ is used throughout instead of Pearson's r — both the predictors and the response have heavy-tailed distributions where the Pearson coefficient is unreliable.

In [4]:
# ---
for col in CONTINUOUS:
    data = df[col].dropna()
    print(f"\n--- {col} ---")
    print(f"  Count: {data.count():,.0f} (missing: {df[col].isna().sum()})")
    print(f"  Mean: {data.mean():,.2f}  Median: {data.median():,.2f}  Std: {data.std():,.2f}")
    print(f"  Min: {data.min():,.2f}  Max: {data.max():,.2f}  Skewness: {data.skew():.4f}")
    print(f"  Zeros: {(data == 0).sum()} ({(data == 0).sum()/len(data)*100:.1f}%)")

# Distributional shape of each continuous predictor: high skewness in sum_insured
# suggests a log-transform may be needed before entering the GLM, to avoid giving
# high-leverage observations disproportionate influence in the linear predictor.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors3 = ['#2196F3', '#4CAF50', '#FF9800']
for idx, col in enumerate(CONTINUOUS):
    data = df[col].dropna()
    axes[idx].hist(data, bins=50, color=colors3[idx], edgecolor='white', alpha=0.85)
    axes[idx].axvline(data.mean(), color='red', ls='--', lw=1.5, label=f'Mean: {data.mean():,.0f}')
    axes[idx].axvline(data.median(), color='orange', ls='--', lw=1.5, label=f'Median: {data.median():,.0f}')
    axes[idx].set_title(col, fontweight='bold'); axes[idx].set_ylabel('Frequency'); axes[idx].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '09_continuous_distributions.png')); plt.close()

# Scatter against log(severity): on raw scale, extreme claims dominate the plot
# and obscure any trend. Spearman ρ is annotated on each panel so the
# predictor-response relationship is immediately legible; the rank-based measure
# captures monotone trends without assuming linearity.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, col in enumerate(CONTINUOUS):
    data = df[[col, TARGET, 'log_claim_severity']].dropna()
    axes[idx].scatter(data[col], data['log_claim_severity'], alpha=0.15, s=8, color=colors3[idx])
    axes[idx].set_xlabel(col); axes[idx].set_ylabel('Log(Claim Severity)')
    axes[idx].set_title(f'{col} vs Log(Severity)', fontweight='bold')
    rho, p = stats.spearmanr(data[col], data[TARGET])
    axes[idx].text(0.05, 0.95, f'Spearman rho={rho:.3f}\np={p:.4f}', transform=axes[idx].transAxes, fontsize=9,
                   va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '10_continuous_vs_severity.png')); plt.close()

# Spearman rank correlation rather than Pearson: robust to the heavy-tailed
# distributions of both predictors and the response; does not require any
# particular marginal distribution.
print(f"\nSpearman Correlations with {TARGET}:")
for col in CONTINUOUS:
    data = df[[col, TARGET]].dropna()
    rho, p = stats.spearmanr(data[col], data[TARGET])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {col:<28} rho={rho:>7.4f}  p={p:.6f}  {sig}")

# Lower-triangle Spearman heatmap across all numeric inputs plus the target.
# Off-diagonal predictor-predictor entries flag collinearity that would inflate
# GLM standard errors; the |r| > 0.7 screen below identifies pairs for removal.
numeric_cols = ['sum_insured_numeric', 'reporting_lag_days', 'coverage_duration_days',
                'is_comprehensive', 'total_claim_severity', 'log_claim_severity']
corr_data = df[numeric_cols].dropna()
corr_matrix = corr_data.corr(method='spearman')
print(f"\nSpearman Correlation Matrix:\n{corr_matrix.round(3).to_string()}")

# |r| > 0.7 as a fast collinearity proxy — the full VIF check runs in the
# GLM notebook. Pairs above this threshold share enough rank-information that
# keeping both adds little independent signal while widening confidence intervals.
print(f"\nMulticollinearity Check (|r| > 0.7):")
predictors = ['sum_insured_numeric', 'reporting_lag_days', 'coverage_duration_days', 'is_comprehensive']
high_corr = False
for i in range(len(predictors)):
    for j in range(i+1, len(predictors)):
        r = corr_matrix.loc[predictors[i], predictors[j]]
        if abs(r) > 0.7:
            print(f"  WARNING: {predictors[i]} & {predictors[j]}: r={r:.3f}"); high_corr = True
if not high_corr: print(f"  No multicollinearity issues detected (all |r| < 0.7)")

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Spearman Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, '11_correlation_matrix.png')); plt.close()


--- sum_insured_numeric ---
  Count: 2,582 (missing: 4)
  Mean: 3,137,038.69  Median: 1,997,500.00  Std: 4,425,864.07
  Min: 0.00  Max: 100,000,000.00  Skewness: 5.9572
  Zeros: 963 (37.3%)

--- reporting_lag_days ---
  Count: 2,371 (missing: 215)
  Mean: 8.61  Median: 1.00  Std: 32.47
  Min: 0.00  Max: 337.00  Skewness: 6.1803
  Zeros: 509 (21.5%)

--- coverage_duration_days ---
  Count: 2,436 (missing: 150)
  Mean: 361.48  Median: 364.00  Std: 66.35
  Min: 1.00  Max: 1,460.00  Skewness: 4.2152
  Zeros: 0 (0.0%)

Spearman Correlations with total_claim_severity:
  sum_insured_numeric          rho= 0.2385  p=0.000000  ***
  reporting_lag_days           rho=-0.1187  p=0.000000  ***
  coverage_duration_days       rho=-0.0438  p=0.030499  *

Spearman Correlation Matrix:
                        sum_insured_numeric  reporting_lag_days  coverage_duration_days  is_comprehensive  total_claim_severity  log_claim_severity
sum_insured_numeric                   1.000              -0.205           

## 4. Predictor Selection and Modelling Setup

Based on the foregoing analysis, I document the keep/drop decision for each candidate predictor. Each decision is grounded in one of three criteria: statistical significance (Kruskal-Wallis H or Spearman ρ), collinearity (|r| > 0.7), or practical considerations (sparse categories, near-constant features).

The final predictor set feeds both the Gamma GLM and XGBoost pipelines under identical 10-fold cross-validation, ensuring the performance comparison is not confounded by different feature inputs.

In [5]:
# ---
decisions = [
    ("class_of_business",      "KEEP",  "Categorical", "Highly significant (p<0.001), 3 clean categories"),
    ("vehicle_type_group",     "KEEP",  "Categorical", "Highly significant (p<0.001), 7->6 groups (merge rare)"),
    ("accident_type",          "KEEP",  "Categorical", "Highly significant (p<0.001), Fire/Overturning differ"),
    ("policy_type",            "DROP",  "Categorical", "Redundant with sum_insured_numeric (SI=0 for TP_Only)"),
    ("is_comprehensive",       "DROP",  "Binary",      "Multicollinear with sum_insured_numeric (r=0.855)"),
    ("branch_name",            "DROP",  "Categorical", "43 branches, small effect (eta2=0.024), 42 dummies"),
    ("sum_insured_numeric",    "KEEP",  "Continuous",  "Strongest predictor (rho=0.237), 4 NaN imputed"),
    ("reporting_lag_days",     "KEEP",  "Continuous",  "Significant (rho=-0.119), 215 NaN imputed"),
    ("coverage_duration_days", "DROP",  "Continuous",  "Near-zero correlation, 79% same value"),
]
print(f"\n  {'Variable':<28} {'Decision':<6} {'Type':<12} Reasoning")
print(f"  {'-'*100}")
for var, dec, typ, reason in decisions:
    marker = "[+]" if dec == "KEEP" else "[-]"
    print(f"  {marker} {var:<25} {dec:<6} {typ:<12} {reason}")

kept = [v for v, d, _, _ in decisions if d == "KEEP"]
print(f"\n  FINAL PREDICTOR SET ({len(kept)} variables): {kept}")

print(f"\n  ML MODEL: XGBoost (reg:gamma objective)")
print(f"  VALIDATION: 10-Fold CV (same folds for both GLM and XGBoost)")
print(f"  METRICS: MAE, RMSE, R2, MAPE, Gamma Deviance")


  Variable                     Decision Type         Reasoning
  ----------------------------------------------------------------------------------------------------
  [+] class_of_business         KEEP   Categorical  Highly significant (p<0.001), 3 clean categories
  [+] vehicle_type_group        KEEP   Categorical  Highly significant (p<0.001), 7->6 groups (merge rare)
  [+] accident_type             KEEP   Categorical  Highly significant (p<0.001), Fire/Overturning differ
  [-] policy_type               DROP   Categorical  Redundant with sum_insured_numeric (SI=0 for TP_Only)
  [-] is_comprehensive          DROP   Binary       Multicollinear with sum_insured_numeric (r=0.855)
  [-] branch_name               DROP   Categorical  43 branches, small effect (eta2=0.024), 42 dummies
  [+] sum_insured_numeric       KEEP   Continuous   Strongest predictor (rho=0.237), 4 NaN imputed
  [+] reporting_lag_days        KEEP   Continuous   Significant (rho=-0.119), 215 NaN imputed
  [-] coverage_